<div style="padding: 20px; background-color: #ffebeb; border-left: 6px solid #d32f2f; font-size: 1.25em;">
<h2>🚨 SECURITY & PUBLISHING WARNING</h2>
If you plan to make this notebook public, <b>DO NOT</b> hardcode your GitHub token. Use <b>Kaggle Secrets</b> instead.<br/><br/>
Also, ensure <code>.virtual_documents</code> is added to your <code>.gitignore</code> before pushing to avoid leaking Kaggle system files.
</div>

<div style="padding: 20px; background-color: #e8f4fd; border-left: 6px solid #1976d2; font-size: 1.25em; margin-top: 10px;">
<h2>⚠️ USAGE INSTRUCTION</h2>
Use <b>Run All</b> <i>only on your first use</i> to reconstruct the repository. After initial setup, run cells individually so you don't overwrite your manual edits!
</div>

# 🛠️ Phase 1: Git Authentication & Setup
---
## 🔑 GitHub Token Setup

Before running this cell:
1. In Kaggle, go to **Add-ons → Secrets** in the top menu
2. Click **Add a new secret**
3. Name it exactly: `GITHUB_TOKEN`
4. Paste your GitHub **fine-grained personal access token** as the value
   - Scope it to your repo only with **Contents: Read and Write** permission
5. Enable the secret for this notebook by toggling it on
6. Then run this cell — your token will never appear in any output

---
**Working on a different branch?** Change `main` to your target branch name in the `git pull` and `git push` cells below.

In [ ]:
!git config --global user.name "YOUR NAME"
!git config --global user.email "YOUR EMAIL"

In [ ]:
from kaggle_secrets import UserSecretsClient
import subprocess
import os

# Fetch your Kaggle Secret (relies on Kaggle Secrets instead of raw YOUR_TOKEN)
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
remote_url = f"https://{token}@github.com/David-Magdy/repo2nb.git"

subprocess.run(["git", "init"], check=True)
subprocess.run(["git", "branch", "-m", "main"], check=True)

try:
    subprocess.run(["git", "remote", "add", "origin", remote_url], check=True, stderr=subprocess.DEVNULL)
except subprocess.CalledProcessError:
    subprocess.run(["git", "remote", "set-url", "origin", remote_url], check=True)

print("Remote URL configured successfully. Token was not printed for security.")

In [ ]:
# Change "main" to your branch name if needed
!git pull origin main

# 📂 Phase 2: Repository Construction
---
The following cells will recreate your project files within Kaggle's environment.

# 📁 

**📄 .gitignore**

In [ ]:
%%writefile /kaggle/working/.gitignore
# Byte-compiled / optimized / DLL files
__pycache__/
*.py[cod]
*$py.class

# C extensions
*.so

# Distribution / packaging
.Python
build/
develop-eggs/
dist/
downloads/
eggs/
.eggs/
lib/
lib64/
parts/
sdist/
var/
wheels/
share/python-wheels/
*.egg-info/
.installed.cfg
*.egg
MANIFEST

# PyInstaller
#  Usually these files are written by a python script from a template
#  before PyInstaller builds the exe, so as to inject date/other infos into it.
*.manifest
*.spec

# Installer logs
pip-log.txt
pip-delete-this-directory.txt

# Unit test / coverage reports
htmlcov/
.tox/
.nox/
.coverage
.coverage.*
.cache
nosetests.xml
coverage.xml
*.cover
*.py,cover
.hypothesis/
.pytest_cache/
cover/

# Jupyter Notebook
.ipynb_checkpoints

# Environments
.env
.venv
env/
venv/
ENV/
env.bak/
venv.bak/



**📄 README.md**

In [ ]:
%%writefile /kaggle/working/README.md
# repo2nb

![repo2nb](logo.png)

repo2nb is an open-source Python CLI tool that converts a local code repository into a self-contained Jupyter notebook (`.ipynb`) designed natively to run on Kaggle's free GPU environment. 

It reconstructs your entire repo in Kaggle's `/kaggle/working` directory using Python's `%%writefile` magic cells, and integrates git for version control so you can pull, test, and push fixes back to your remote repository without ever leaving the Kaggle interface.

## Motivation
*This is a tool I made for personal use first, then I wanted to publish it.*

My motivation was that I wanted to securely run a training repo on Kaggle, but it was scattered across directories and Python files. It is extremely frustrating copying all of this into a notebook and debugging why it's giving an error. 

I used to do workarounds like uploading the repo as a dataset and starting from there. It was exhausting and wasted a couple of minutes just to realize I missed an indented line! Attempting the same flow manually using git for authenticating myself, pushing, and pulling for simple microscopic changes, which was equally painful. `repo2nb` automates all of this seamlessly.

**Heads Up:** This project is intended for personal and academic projects. It is specifically designed for students and hobbyists like myself who want to quickly leverage free GPU compute without friction, rather than managing massive corporate repositories with hundreds of nested files!

## Installation

```bash
pip install repo2nb
```

## Usage

```bash
# Convert your local project to a Kaggle notebook
repo2nb ./my_project --output my_project_kaggle.ipynb
```

Then literally just upload the resulting `.ipynb` file to Kaggle!

### Options

- `--output`, `-o`: Specify the output notebook path. Defaults to `<folder_name>.ipynb`.
- `--omit-instructions`: Omits the beginner-friendly warning cells and instructional git cheat sheets. Perfect for power users who already know the setup routine.
- `--ignore-extra`: Provide extra file extensions to ignore entirely (e.g., `--ignore-extra ".yaml .json"`). These files will be completely omitted from the generated notebook.
- `--include`: Force include specific file extensions that are usually skipped as binary/data (e.g., `--include ".csv .json"`).

## Features

- **Instant Rebuild**: Automatically translates your local file tree into correctly ordered `%%writefile` blocks. 
- **Git Integration**: Injects pre-formatted shell cells for initializing Git, adding tokens, selecting branches, pulling, and pushing.
- **Smart Filtering**: Automatically skips cached data, virtual environments (`.venv`), `uv` lock files, heavy binaries (`.pt`, `.pkl`, `.jpg`), and dataset files (`.csv`, `.parquet`) so your final notebook remains incredibly lightweight.
- **Visual Segregation**: Creates unmissable structural phases isolating where the automatic repo build ends and where your actual coding workspace begins.
- **Built-in Git Cheat Sheet**: Gives you immediate interactive access to `git status`, `git rm -rf`, and `git mv` blocks directly in the UI.

## Important Conventions & Security

**Security & Publishing:**
`repo2nb` uses Kaggle's native secrets manager to inject your GitHub token at runtime. Your token is never hardcoded, never visible in cell output, and the notebook remains safe to share publicly. Additionally, ensure that Kaggle's `.virtual_documents` directory is added to your `.gitignore` before pushing any changes to avoid leaking Kaggle's system files into your repo.

**Run All Only Once:**
When you first start your Kaggle session, use **"Run All"** to bootstrap the directory structure and recreate the files.
**After the initial setup, run cells individually as needed.** Using "Run All" again may overwrite any local manual code changes you have made that session!

**Branch Management:**
The notebook's git hooks default to `main`. The generated code blocks remind you to swap `"main"` for your target branch name if you are pulling or committing to a different branch.


**Skipped data/binary file**: `logo.png`
*(Upload manually if needed)*

**Skipped data/binary file**: `rl.ipynb`
*(Upload manually if needed)*

### 📁 workflows

**📄 ci.yml**

In [ ]:
%%writefile /kaggle/working/.github/workflows/ci.yml
name: CI

on:
  push:
    branches: ["*"]
  pull_request:
    branches: ["master"]

jobs:
  lint-and-test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v5

      - name: Install uv
        uses: astral-sh/setup-uv@v6

      - name: Install dependencies
        run: uv sync --all-extras --dev

      - name: Lint with ruff
        run: uv run ruff check .

      - name: Run tests
        run: uv run pytest tests/


## 📁 .ruff_cache

**📄 .gitignore**

In [ ]:
%%writefile /kaggle/working/.ruff_cache/.gitignore
# Automatically created by ruff.
*


**📄 CACHEDIR.TAG**

In [ ]:
%%writefile /kaggle/working/.ruff_cache/CACHEDIR.TAG
Signature: 8a477f597d28d172789f06886806bc55

### 📁 0.15.10

**📄 11324829422823581138**

**Skipped non-UTF8 file**: `.ruff_cache/0.15.10/11324829422823581138`
*(Upload manually if needed)*

**📄 15989689542345452**

**Skipped non-UTF8 file**: `.ruff_cache/0.15.10/15989689542345452`
*(Upload manually if needed)*

**📄 2406084278179533104**

**Skipped non-UTF8 file**: `.ruff_cache/0.15.10/2406084278179533104`
*(Upload manually if needed)*

## 📁 examples

**Skipped data/binary file**: `examples/sample_output.ipynb`
*(Upload manually if needed)*

**Skipped data/binary file**: `examples/sample_output_omit_instructions.ipynb`
*(Upload manually if needed)*

## 📁 repo2nb

**📄 __init__.py**

In [ ]:
%%writefile /kaggle/working/repo2nb/__init__.py
"""repo2nb: Convert local repository into a Kaggle Notebook."""


**📄 cli.py**

In [ ]:
%%writefile /kaggle/working/repo2nb/cli.py
import click
import pathlib
from .converter import convert

@click.command()
@click.argument("repo_path", type=click.Path(exists=True, file_okay=False, dir_okay=True, path_type=pathlib.Path))
@click.option("--output", "-o", type=click.Path(dir_okay=False, path_type=pathlib.Path), help="Output .ipynb path")
@click.option("--omit-instructions", is_flag=True, default=False, help="Omit warning cells and instructional cheat sheets.")
@click.option("--ignore-extra", type=str, help="Extra file extensions to ignore completely (e.g. '.mp4 .yaml').")
@click.option("--include", type=str, help="File extensions to force include (e.g. '.csv .json').")
def main(repo_path, output, omit_instructions, ignore_extra, include):
    """Convert a local directory into a reproducible Jupyter Notebook for Kaggle."""
    if not output:
        output = pathlib.Path(f"{repo_path.name}.ipynb")
    
    ignore_set = set(ignore_extra.replace(",", " ").split()) if ignore_extra else set()
    include_set = set(include.replace(",", " ").split()) if include else set()
    
    convert(repo_path, output, omit_instructions, ignore_set, include_set)

if __name__ == "__main__":
    main()


**📄 converter.py**

In [ ]:
%%writefile /kaggle/working/repo2nb/converter.py
import pathlib
import nbformat
from .traversal import traverse
from .warnings import get_warning_cells
from .git import get_git_cells
from .notebook import make_markdown_cell, make_writefile_cell, assemble_notebook

def _is_binary(file_path: pathlib.Path, include: set = None) -> bool:
    binary_extensions = {
        '.pkl', '.pt', '.h5', '.png', '.jpg', '.jpeg', '.gif', '.zip', '.tar', '.gz',
        '.csv', '.tsv', '.xlsx', '.xls', '.parquet', '.db', '.sqlite', '.pdf', '.ipynb', 
    }
    if include:
        include_exts = {ext.lower() if ext.startswith('.') else '.' + ext.lower() for ext in include}
        binary_extensions = binary_extensions - include_exts
    return file_path.suffix.lower() in binary_extensions

def convert(repo_path: pathlib.Path, output_path: pathlib.Path, omit_instructions: bool = False, ignore_extra: set = None, include: set = None):
    ignore_extra = ignore_extra or set()
    ignore_exts = {ext.lower() if ext.startswith('.') else '.' + ext.lower() for ext in ignore_extra}
    
    tree, has_git = traverse(repo_path)
    
    cells = []
    
    if not omit_instructions:
        cells.extend(get_warning_cells())
    
    if has_git:
        setup_cells, push_cells = get_git_cells(repo_path, omit_instructions)
        cells.extend(setup_cells)
        
    if omit_instructions:
        phase2_text = "# 📂 Phase 2: Repository Construction"
    else:
        phase2_text = (
            "# 📂 Phase 2: Repository Construction\n"
            "---\n"
            "The following cells will recreate your project files within Kaggle's environment."
        )
    cells.append(make_markdown_cell(phase2_text))

    repo_name = repo_path.name

    for dir_path, files in tree:
        valid_files = [f for f in files if f.suffix.lower() not in ignore_exts]
        if not valid_files:
            continue
            

        if dir_path == repo_path:
            depth = 0
            folder_name = repo_name
        else:
            try:
                rel_dir = dir_path.relative_to(repo_path)
                depth = len(rel_dir.parts)
            except ValueError:
                depth = 0
            folder_name = dir_path.name
            
        if depth == 0:
            header_level = "#"
        elif depth == 1:
            header_level = "##"
        elif depth == 2:
            header_level = "###"
        else:
            header_level = "####"
            
        cells.append(make_markdown_cell(f"{header_level} 📁 {folder_name}"))
        
        for file_path in valid_files:
            try:
                rel_file_path = file_path.relative_to(repo_path)
            except ValueError:
                continue
                
            kag_path = rel_file_path.as_posix()
            
            if _is_binary(file_path, include):
                cells.append(make_markdown_cell(f"**Skipped data/binary file**: `{kag_path}`\n*(Upload manually if needed)*"))
                continue
                
            cells.append(make_markdown_cell(f"**📄 {rel_file_path.name}**"))
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    content = f.read()
                cells.append(make_writefile_cell(kag_path, content))
            except UnicodeDecodeError:
                cells.append(make_markdown_cell(f"**Skipped non-UTF8 file**: `{kag_path}`\n*(Upload manually if needed)*"))
                
    if has_git:
        cells.extend(push_cells)
        
    nb = assemble_notebook(cells)
    
    with open(output_path, "w", encoding="utf-8") as f:
        nbformat.write(nb, f)


**📄 git.py**

In [ ]:
%%writefile /kaggle/working/repo2nb/git.py
import pathlib
import nbformat.v4 as nbf

def _get_remote_url(repo_path: pathlib.Path) -> str:
    config_path = repo_path / ".git" / "config"
    if config_path.exists():
        try:
            with open(config_path, "r", encoding="utf-8") as f:
                content = f.read()
                in_origin = False
                for line in content.splitlines():
                    line = line.strip()
                    if line == '[remote "origin"]':
                        in_origin = True
                    elif line.startswith('[') and in_origin:
                        in_origin = False
                    elif in_origin and line.startswith('url ='):
                        url = line.split('=', 1)[1].strip()
                        if "github.com" in url:
                            suffix = url.split("github.com")[-1].lstrip(":/").rstrip("/")
                            if suffix.endswith(".git"):
                                return suffix
                            return suffix + ".git"
                        return url
        except Exception:
            pass
    return "user/repo.git"

def get_git_cells(repo_path: pathlib.Path, omit_instructions: bool = False):
    remote_suffix = _get_remote_url(repo_path)
    
    if omit_instructions:
        setup_md_text = "# 🛠️ Phase 1: Git Authentication & Setup"
    else:
        setup_md_text = (
            "# 🛠️ Phase 1: Git Authentication & Setup\n"
            "---\n"
            "## 🔑 GitHub Token Setup\n\n"
            "Before running this cell:\n"
            "1. In Kaggle, go to **Add-ons → Secrets** in the top menu\n"
            "2. Click **Add a new secret**\n"
            "3. Name it exactly: `GITHUB_TOKEN`\n"
            "4. Paste your GitHub **fine-grained personal access token** as the value\n"
            "   - Scope it to your repo only with **Contents: Read and Write** permission\n"
            "5. Enable the secret for this notebook by toggling it on\n"
            "6. Then run this cell — your token will never appear in any output\n\n"
            "---\n"
            "**Working on a different branch?** Change `main` to your target branch name in the `git pull` and `git push` cells below."
        )
    setup_md = nbf.new_markdown_cell(setup_md_text)
    
    config_code = '!git config --global user.name "YOUR NAME"\n!git config --global user.email "YOUR EMAIL"'
    config_cell = nbf.new_code_cell(config_code)
    
    remote_code = (
        'from kaggle_secrets import UserSecretsClient\n'
        'import subprocess\n'
        'import os\n\n'
        '# Fetch your Kaggle Secret (relies on Kaggle Secrets instead of raw YOUR_TOKEN)\n'
        'token = UserSecretsClient().get_secret("GITHUB_TOKEN")\n'
        f'remote_url = f"https://{{token}}@github.com/{remote_suffix}"\n\n'
        'subprocess.run(["git", "init"], check=True)\n'
        'subprocess.run(["git", "branch", "-m", "main"], check=True)\n\n'
        'try:\n'
        '    subprocess.run(["git", "remote", "add", "origin", remote_url], check=True, stderr=subprocess.DEVNULL)\n'
        'except subprocess.CalledProcessError:\n'
        '    subprocess.run(["git", "remote", "set-url", "origin", remote_url], check=True)\n\n'
        'print("Remote URL configured successfully. Token was not printed for security.")'
    )
    remote_cell = nbf.new_code_cell(remote_code)
    
    pull_code = '# Change "main" to your branch name if needed\n!git pull origin main'
    pull_cell = nbf.new_code_cell(pull_code)
    
    setup_cells = [setup_md, config_cell, remote_cell, pull_cell]
    
    push_code = (
        '# Un-comment the lines below when you are ready to push!\n'
        '# !git add .\n'
        '# !git commit -m "fix from kaggle session"\n'
        '# !git push origin main'
    )
    push_cell = nbf.new_code_cell(push_code)
    
    if omit_instructions:
        push_cells = [nbf.new_markdown_cell("# 🚀 Phase 3: Your Workspace"), push_cell]
    else:
        cheat_sheet_md = nbf.new_markdown_cell(
            "# 🚀 Phase 3: Your Workspace\n"
            "---\n"
            "### <span style='color: #2e7d32;'>**Start manipulating and running your code from here onwards!**</span>\n\n"
            "## Git Cheat Sheet\n"
            "Uncomment the cell below when you are ready to push. Other useful commands:\n"
            "- **Remove a file**: `!git rm path/to/file.ext`\n"
            "- **Remove a folder**: `!git rm -rf path/to/folder`\n"
            "- **Rename a file**: `!git mv old_name.ext new_name.ext`\n"
            "- **Check status**: `!git status`"
        )
        push_cells = [cheat_sheet_md, push_cell]
        
    return setup_cells, push_cells


**📄 notebook.py**

In [ ]:
%%writefile /kaggle/working/repo2nb/notebook.py
import nbformat.v4 as nbf
import nbformat

def make_markdown_cell(text: str) -> nbformat.NotebookNode:
    return nbf.new_markdown_cell(text)

def make_code_cell(code: str) -> nbformat.NotebookNode:
    return nbf.new_code_cell(code)

def make_writefile_cell(filepath: str, content: str) -> nbformat.NotebookNode:
    code = f"%%writefile /kaggle/working/{filepath}\n{content}"
    return make_code_cell(code)

def assemble_notebook(cells: list) -> nbformat.NotebookNode:
    nb = nbf.new_notebook()
    nb.cells = cells
    return nb


**📄 traversal.py**

In [ ]:
%%writefile /kaggle/working/repo2nb/traversal.py
import pathlib
import fnmatch

DEFAULT_IGNORE = [
    "__pycache__", "*.pyc", "node_modules", ".DS_Store", "venv", ".venv", ".env", "*.egg-info", "dist", "build", ".git", ".pytest_cache", ".hypothesis", ".coverage",
    "uv.lock", ".python-version", "pyproject.toml"
]

def should_ignore(path: pathlib.Path) -> bool:
    name = path.name
    for pattern in DEFAULT_IGNORE:
        if fnmatch.fnmatch(name, pattern):
            return True
    return False

def traverse(root_path: pathlib.Path) -> tuple[list, bool]:
    has_git = (root_path / ".git").is_dir()
    
    tree = []
    
    def _walk(current_path: pathlib.Path):
        if should_ignore(current_path):
            return
            
        files = []
        dirs = []
        for child in sorted(current_path.iterdir()):
            if should_ignore(child):
                continue
            if child.is_dir():
                dirs.append(child)
            else:
                files.append(child)
                
        tree.append((current_path, files))
        for d in dirs:
            _walk(d)
            
    _walk(root_path)
    return tree, has_git


**📄 warnings.py**

In [ ]:
%%writefile /kaggle/working/repo2nb/warnings.py
import nbformat.v4 as nbf

def get_warning_cells():
    security_text = (
        "<div style=\"padding: 20px; background-color: #ffebeb; border-left: 6px solid #d32f2f; font-size: 1.25em;\">\n"
        "<h2>🚨 SECURITY & PUBLISHING WARNING</h2>\n"
        "If you plan to make this notebook public, <b>DO NOT</b> hardcode your GitHub token. Use <b>Kaggle Secrets</b> instead.<br/><br/>\n"
        "Also, ensure <code>.virtual_documents</code> is added to your <code>.gitignore</code> before pushing to avoid leaking Kaggle system files.\n"
        "</div>"
    )
    usage_text = (
        "<div style=\"padding: 20px; background-color: #e8f4fd; border-left: 6px solid #1976d2; font-size: 1.25em; margin-top: 10px;\">\n"
        "<h2>⚠️ USAGE INSTRUCTION</h2>\n"
        "Use <b>Run All</b> <i>only on your first use</i> to reconstruct the repository. After initial setup, run cells individually so you don't overwrite your manual edits!\n"
        "</div>"
    )
    
    return [
        nbf.new_markdown_cell(security_text),
        nbf.new_markdown_cell(usage_text)
    ]


## 📁 tests

**📄 test_converter.py**

In [ ]:
%%writefile /kaggle/working/tests/test_converter.py
import nbformat
from repo2nb.converter import convert

def test_converter_no_git(tmp_path):
    repo_path = tmp_path / "my_project"
    repo_path.mkdir()
    (repo_path / "test.py").write_text("print('hello')")
    
    output_path = tmp_path / "out.ipynb"
    convert(repo_path, output_path)
    
    assert output_path.exists()
    
    with open(output_path, "r") as f:
        nb = nbformat.read(f, as_version=4)
        
    cells = nb.cells
    assert len(cells) == 6
    
    header_cell = cells[3]
    assert header_cell.cell_type == "markdown"
    assert header_cell.source == "# 📁 my_project"
    
    file_title_cell = cells[4]
    assert file_title_cell.cell_type == "markdown"
    assert "test.py" in file_title_cell.source
    
    code_cell = cells[5]
    assert code_cell.cell_type == "code"
    assert "%%writefile" in code_cell.source
    assert "/kaggle/working/test.py" in code_cell.source

def test_converter_depth(tmp_path):
    repo_path = tmp_path / "my_project"
    repo_path.mkdir()
    (repo_path / "test0.py").touch()
    
    sub_dir = repo_path / "level1"
    sub_dir.mkdir()
    (sub_dir / "test1.py").touch()
    
    sub_sub_dir = sub_dir / "level2"
    sub_sub_dir.mkdir()
    (sub_sub_dir / "test2.py").touch()
    
    output_path = tmp_path / "out.ipynb"
    convert(repo_path, output_path)
    
    with open(output_path, "r") as f:
        nb = nbformat.read(f, as_version=4)
        
    markdown_sources = [c.source for c in nb.cells if c.cell_type == "markdown"]
    assert "# 📁 my_project" in markdown_sources
    assert "## 📁 level1" in markdown_sources
    assert "### 📁 level2" in markdown_sources



**📄 test_git.py**

In [ ]:
%%writefile /kaggle/working/tests/test_git.py
from repo2nb.git import get_git_cells, _get_remote_url

def test_git_cells(tmp_path):
    setup_cells, push_cells = get_git_cells(tmp_path)
    
    assert len(setup_cells) == 4
    assert len(push_cells) == 2
    
    config_code = setup_cells[1].source
    assert "YOUR NAME" in config_code
    assert "YOUR EMAIL" in config_code
    
    remote_code = setup_cells[2].source
    assert "YOUR_TOKEN" in remote_code
    assert "user/repo.git" in remote_code

def test_get_remote_url(tmp_path):
    git_dir = tmp_path / ".git"
    git_dir.mkdir()
    config = git_dir / "config"
    config.write_text('[remote "origin"]\n\turl = https://github.com/myuser/myrepo.git\n')
    
    url = _get_remote_url(tmp_path)
    assert url == "myuser/myrepo.git"


**📄 test_notebook.py**

In [ ]:
%%writefile /kaggle/working/tests/test_notebook.py
from repo2nb.notebook import make_writefile_cell, assemble_notebook, make_markdown_cell

def test_make_writefile_cell():
    cell = make_writefile_cell("my_project/test.py", "print('hello')")
    assert cell.cell_type == "code"
    assert cell.source.startswith("%%writefile /kaggle/working/my_project/test.py\nprint('hello')")

def test_assemble_notebook():
    cells = [make_markdown_cell("Test")]
    nb = assemble_notebook(cells)
    
    assert nb.nbformat == 4
    assert len(nb.cells) == 1
    assert nb.cells[0].source == "Test"


**📄 test_traversal.py**

In [ ]:
%%writefile /kaggle/working/tests/test_traversal.py
import pathlib
from repo2nb.traversal import traverse, should_ignore

def test_should_ignore():
    assert should_ignore(pathlib.Path("__pycache__"))
    assert should_ignore(pathlib.Path("node_modules"))
    assert should_ignore(pathlib.Path("test.pyc"))
    assert not should_ignore(pathlib.Path("test.py"))
    assert should_ignore(pathlib.Path(".git"))

def test_traverse_tree(tmp_path):
    (tmp_path / ".git").mkdir()
    (tmp_path / "file1.py").touch()
    
    folder1 = tmp_path / "folder1"
    folder1.mkdir()
    (folder1 / "file2.py").touch()
    
    pycache = tmp_path / "__pycache__"
    pycache.mkdir()
    (pycache / "ignored.pyc").touch()
    
    tree, has_git = traverse(tmp_path)
    
    assert has_git is True
    assert len(tree) == 2
    
    # Root level
    assert tree[0][0] == tmp_path
    assert len(tree[0][1]) == 1
    assert tree[0][1][0].name == "file1.py"
    
    # Sub folder
    assert tree[1][0] == folder1
    assert len(tree[1][1]) == 1
    assert tree[1][1][0].name == "file2.py"


# 🚀 Phase 3: Your Workspace
---
### <span style='color: #2e7d32;'>**Start manipulating and running your code from here onwards!**</span>

## Git Cheat Sheet
Uncomment the cell below when you are ready to push. Other useful commands:
- **Remove a file**: `!git rm path/to/file.ext`
- **Remove a folder**: `!git rm -rf path/to/folder`
- **Rename a file**: `!git mv old_name.ext new_name.ext`
- **Check status**: `!git status`

In [ ]:
# Un-comment the lines below when you are ready to push!
# !git add .
# !git commit -m "fix from kaggle session"
# !git push origin main